In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

# Pre-Processing

## 1. Converting to lowercase

In [7]:
df["review"]=df["review"].str.lower()

## 2. Removing the URLs

In [8]:
# example use for re library
import re

sample_text = "abc is the word, abc" # abc=> xyx
re.sub("abc","xyz", sample_text)

'xyz is the word, xyz'

In [9]:
def remove_urls(text):
    #\S - non-white space character
    #\s - white space character
    text =re.sub(r"http\S+", "", text) # (pattern , repl , string)
    return text

df["review"] = df["review"].apply(remove_urls)

## 3. Removing punchuations

In [10]:
def remove_puntuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) # A-Z  a-z 0-9 \s
    return text
df["review"] = df["review"].apply(remove_puntuations)

In [11]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


## 4. Removing HTML

In [12]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)
    

## 5. Removing the Stopwords

In [13]:
import nltk
# natural languagr toolkit

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
sample_text = " I like coding in python!"
tokens = word_tokenize(sample_text)

In [16]:
tokens

['I', 'like', 'coding', 'in', 'python', '!']

In [17]:
def remove_stopwords(text):
    token = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in token:
        if word in stop_words:
            tex = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)


In [18]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 6. Stemming

In [19]:
# running -> run
# played -> play
# PorterStemming

In [20]:
from nltk.stem import PorterStemmer
def stemming(text):
    ps = PorterStemmer()
    stemmed_words= []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)
df["review"] = df["review"].apply(stemming)


### 7. Encoding

In [23]:
from sklearn.preprocessing import LabelEncoder

# ✅ Initialize encoder
le = LabelEncoder()

# ✅ Fit and transform sentiment column
df["sentiment"] = le.fit_transform(df["sentiment"])


In [24]:
y = df["sentiment"]

In [25]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int32

### 8. Vectorization

In [26]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,1
1,a wonder littl product br br the film techniqu...,1
2,i thought thi wa a wonder way to spend time on...,1
3,basic there a famili where a littl boy jake th...,0
4,petter mattei love in the time of money is a v...,1


In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [30]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5871102 stored elements and shape (49582, 5000)>

## Dataet & Data Loaders

In [47]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [48]:
X_train.shape

(39665, 5000)

In [49]:
X_test.shape

(9917, 5000)

In [50]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [51]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4699371 stored elements and shape (39665, 5000)>

In [52]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [53]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [54]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [57]:
import torch.nn as nn
import torch.optim as optim

In [62]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN Layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer 
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ =self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [63]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())


## Training the RNN

In [66]:
# unsqueeze squeeze

epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze())# (batch_size, ) => probability

        loss = criterion(outputs, yb) #compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch}/{epochs} and loss = {loss.item()}")

epoch = 0/10 and loss = 0.6440606117248535
epoch = 1/10 and loss = 0.2604588270187378
epoch = 2/10 and loss = 0.1528400182723999
epoch = 3/10 and loss = 0.2497974932193756
epoch = 4/10 and loss = 0.4095400869846344
epoch = 5/10 and loss = 0.16372118890285492
epoch = 6/10 and loss = 0.08306171745061874
epoch = 7/10 and loss = 0.14894215762615204
epoch = 8/10 and loss = 0.20093902945518494
epoch = 9/10 and loss = 0.19362285733222961


In [70]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5 ).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 87.42563275184028
